# Notebook 03: ASL Model Training, Validation & Evaluation

Notebook này tải dữ liệu landmarks đã trích xuất từ Notebook 02, áp dụng chuẩn hóa không gian và thời gian, huấn luyện mô hình PyTorch GRU và đánh giá toàn diện trên tập test độc lập.

### Nội dung:
1. **Load landmarks**: Đọc manifest `extracted_landmarks.csv` và kiểm tra GPU.
2. **Normalization**: Chuẩn hóa không gian đưa cổ tay về gốc tọa độ và scale bất biến kích thước.
3. **Temporal resampling**: Resampling chuỗi thời gian về độ dài cố định $T = 32$ frames.
4. **Dataset/DataLoader**: Tạo PyTorch Dataset & DataLoader cho train, val, test.
5. **GRU**: Xây dựng kiến trúc mô hình PyTorch GRU Classifier.
6. **Training**: Vòng lặp huấn luyện với CrossEntropyLoss, Adam Optimizer và ReduceLROnPlateau.
7. **Validation**: Theo dõi Loss và Accuracy qua từng epoch.
8. **Test**: Đánh giá checkpoint tốt nhất trên tập test độc lập.
9. **Confusion matrix**: Vẽ và xuất ma trận nhầm lẫn và biểu đồ học tập.
10. **Save best checkpoint**: Lưu model trọng số (`best_model.pth`) để sử dụng cho realtime demo.


---
## 1. Setup Môi trường & Load Manifest


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name    : {torch.cuda.get_device_name(0)}")

IN_KAGGLE = Path('/kaggle/working').exists()
OUTPUT_ROOT = Path('/kaggle/working/asl_outputs') if IN_KAGGLE else Path('outputs').resolve()
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
PLOTS_OUTPUT_DIR = OUTPUT_ROOT / 'plots'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = OUTPUT_ROOT / 'extracted_landmarks.csv'
if not manifest_path.exists():
    raise FileNotFoundError(f"Không tìm thấy {manifest_path}. Hãy chạy Notebook 02 để trích xuất landmarks trước!")

df_manifest = pd.read_csv(manifest_path)
print(f"Loaded manifest with {len(df_manifest)} samples across splits:")
print(df_manifest['split'].value_counts())


---
## 2. Normalization & Temporal Resampling


In [ ]:
def normalize_hand_coords(hand_pts):
    """Dời cổ tay (index 0) về gốc tọa độ và scale theo khoảng cách cổ tay - gốc ngón giữa (index 9)."""
    if np.all(hand_pts == 0):
        return hand_pts
    wrist = hand_pts[0:1, :]
    translated = hand_pts - wrist
    scale = np.linalg.norm(translated[9] - translated[0])
    return translated / (scale if scale > 1e-6 else 1.0)

def normalize_frame(features_126):
    """Chuẩn hóa độc lập bàn tay trái [0:63] và bàn tay phải [63:126]."""
    left = normalize_hand_coords(features_126[:63].reshape(21, 3)).flatten()
    right = normalize_hand_coords(features_126[63:].reshape(21, 3)).flatten()
    return np.concatenate([left, right], axis=0)

def resample_temporal(seq, target_len=32):
    """Uniform temporal resampling về đúng target_len frames."""
    t = len(seq)
    if t == 0:
        return np.zeros((target_len, 126), dtype=np.float32)
    if t == target_len:
        return seq.astype(np.float32)
    indices = np.linspace(0, t - 1, target_len)
    resampled = np.zeros((target_len, seq.shape[1]), dtype=np.float32)
    for col in range(seq.shape[1]):
        resampled[:, col] = np.interp(indices, np.arange(t), seq[:, col])
    return resampled

print("Normalization & Temporal Resampling functions defined!")


---
## 3. PyTorch Dataset & DataLoader


In [ ]:
class ASLLandmarkDataset(Dataset):
    def __init__(self, df, class_to_idx, target_len=32):
        self.samples = df.to_dict('records')
        self.class_to_idx = class_to_idx
        self.target_len = target_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        raw_seq = np.load(item['npy_path'])
        # 1. Spatial normalization từng frame
        norm_seq = np.array([normalize_frame(f) for f in raw_seq], dtype=np.float32)
        # 2. Temporal uniform resampling về 32 frames
        fixed_seq = resample_temporal(norm_seq, target_len=self.target_len)
        label = self.class_to_idx[item['gloss']]
        return torch.tensor(fixed_seq, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

unique_classes = sorted(df_manifest['gloss'].unique().tolist())
class_to_idx = {c: i for i, c in enumerate(unique_classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
print("Class Mapping:", class_to_idx)

df_train = df_manifest[df_manifest['split'] == 'train']
df_val = df_manifest[df_manifest['split'] == 'val']
df_test = df_manifest[df_manifest['split'] == 'test']

print(f"Splits: Train = {len(df_train)} | Val = {len(df_val)} | Test = {len(df_test)}")

BATCH_SIZE = 32
train_loader = DataLoader(ASLLandmarkDataset(df_train, class_to_idx), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ASLLandmarkDataset(df_val, class_to_idx), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(ASLLandmarkDataset(df_test, class_to_idx), batch_size=BATCH_SIZE, shuffle=False)


---
## 4. PyTorch GRU Classifier Architecture


In [ ]:
class ASLGRUClassifier(nn.Module):
    def __init__(self, input_size=126, hidden_size=128, num_layers=2, num_classes=5, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        # x shape: [B, 32, 126]
        _, h_n = self.gru(x)
        last_hidden = h_n[-1]  # [B, hidden_size]
        logits = self.classifier(last_hidden)
        return logits

model = ASLGRUClassifier(num_classes=len(unique_classes)).to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable Parameters: {n_params:,}")


---
## 5. Training Loop & Validation


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

EPOCHS = 40
best_val_loss = float('inf')
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print(f"Bắt đầu training trên {device} trong {EPOCHS} epochs...")

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    t_loss, t_corr, t_total = 0.0, 0, 0
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        preds = model(x_b)
        loss = criterion(preds, y_b)
        loss.backward()
        optimizer.step()
        
        t_loss += loss.item() * x_b.size(0)
        t_corr += (preds.argmax(dim=1) == y_b).sum().item()
        t_total += y_b.size(0)
        
    train_loss = t_loss / max(t_total, 1)
    train_acc = t_corr / max(t_total, 1)
    
    # --- Validation ---
    model.eval()
    v_loss, v_corr, v_total = 0.0, 0, 0
    with torch.no_grad():
        for x_b, y_b in val_loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            preds = model(x_b)
            loss = criterion(preds, y_b)
            v_loss += loss.item() * x_b.size(0)
            v_corr += (preds.argmax(dim=1) == y_b).sum().item()
            v_total += y_b.size(0)
            
    val_loss = v_loss / max(v_total, 1)
    val_acc = v_corr / max(v_total, 1)
    
    scheduler.step(val_loss)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] Train Loss: {train_loss:.4f} | Acc: {train_acc:.3f} | Val Loss: {val_loss:.4f} | Acc: {val_acc:.3f}")
        
    # Lưu best model checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CHECKPOINT_DIR / 'best_model.pth')

# Luôn lưu checkpoint cuối cùng
torch.save(model.state_dict(), CHECKPOINT_DIR / 'last_model.pth')
print(f"\nTraining hoàn tất! Checkpoint tốt nhất đã được lưu tại: {CHECKPOINT_DIR / 'best_model.pth'}")


---
## 6. Test Set Evaluation & Confusion Matrix


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

# Load lại best model checkpoint
model.load_state_dict(torch.load(CHECKPOINT_DIR / 'best_model.pth', map_location=device))
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for x_b, y_b in test_loader:
        x_b = x_b.to(device)
        preds = model(x_b).argmax(dim=1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(y_b.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("=" * 60)
print("TEST SET CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=unique_classes, zero_division=0))

# Biểu đồ Loss & Accuracy curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['train_acc'], label='Train Accuracy')
axes[1].plot(history['val_acc'], label='Val Accuracy')
axes[1].set_title('Training & Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.savefig(PLOTS_OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=np.arange(len(unique_classes)),
    yticks=np.arange(len(unique_classes)),
    xticklabels=unique_classes,
    yticklabels=unique_classes,
    title='Test Set Confusion Matrix',
    ylabel='True Label',
    xlabel='Predicted Label'
)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
for i in range(len(unique_classes)):
    for j in range(len(unique_classes)):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.savefig(PLOTS_OUTPUT_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Model Checkpoint sẵn sàng tải về tại: {CHECKPOINT_DIR / 'best_model.pth'}")
